# phase 3 brrrt — layer-0 probe direction under vLLM + vllm-lens, teacher-forced extraction, HF upload
Inputs uploaded to `/content`: `probe_l0_direction.json`, `job_probe_l0_vllm.py`.

In [ ]:
# === CELL 1 — install (phase 1's versions) ==========================================================
import subprocess, sys, time; t0 = time.time()
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.29.0", "vllm-lens==1.2.1"], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-3000:]); print(f"{time.time()-t0:.0f}s")
import importlib; print("vllm", importlib.metadata.version("vllm"), "| vllm-lens", importlib.metadata.version("vllm-lens"), "| torch", importlib.metadata.version("torch"))


In [ ]:
# === CELL 2 — run the job as a subprocess (frees the GPU when it exits) =============================
import subprocess, os, time
env = dict(os.environ, OUT_DIR="/content/brrrt", MINUTES="30", MAX_NUM_SEQS="256", SEEDS_PER_ROUND="16", CLEAN_SEEDS="8")
with open("/content/brrrt_job.log", "w") as f:
    p = subprocess.Popen(["python3", "-u", "/content/job_probe_l0_vllm.py"], stdout=f, stderr=subprocess.STDOUT, env=env)
print("started pid", p.pid)


In [ ]:
# === CELL 3 — poll the job log ======================================================================
import subprocess, time, json, os
print(subprocess.run(["tail", "-n", "25", "/content/brrrt_job.log"], capture_output=True, text=True).stdout)
print("gen.jsonl lines:", sum(1 for _ in open("/content/brrrt/gen.jsonl")) if os.path.exists("/content/brrrt/gen.jsonl") else 0)


In [ ]:
# === CELL 4 — teacher-forced passes to the 8th token, residual stream at every 4th layer (HF, phase 3's hook) ===
import torch, json, time, os, numpy as np, inspect
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = "Qwen/Qwen3-8B"; DIR = json.load(open("/content/probe_l0_direction.json"))
U = torch.tensor(DIR["u"]); U = (U / U.norm()); EPS = float(DIR["eps"])
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda:0"); model.eval(); model.requires_grad_(False)
dev = model.device; EOS = tokenizer.eos_token_id; U = U.to(dev)
ROLL = [json.loads(l) for l in open("/content/brrrt/gen.jsonl")]
print(len(ROLL), "rollouts |", sum(r["arm"] == "clean" for r in ROLL), "clean")
STATE = {"mask": None, "on": False}
def _hook(mod, inp, out):
    if not STATE["on"]: return out
    hs = out[0] if isinstance(out, tuple) else out; B, T, _ = hs.shape
    m = STATE["mask"][:B, :T].to(hs.dtype)[..., None]; hf = hs.float(); nrm = hf.norm(dim=-1, keepdim=True)
    new = (hf + EPS * nrm * U.float()[None, None, :] * m.float()).to(hs.dtype)
    return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new
H = model.model.layers[0].register_forward_hook(_hook)
LAYERS = list(range(0, model.config.num_hidden_layers + 1, 4)); K = 8; SLOTS = ["P"] + [f"R{k}" for k in range(1, K + 1)]; D = model.config.hidden_size
BS = 48; SHARD = 2000; os.makedirs("/content/brrrt/features", exist_ok=True)
# rig check: clean H1 for Q and direction H1 for Q from the same forward used below
def h1(ids, on):
    STATE["on"] = on; STATE["mask"] = torch.ones((1, len(ids)), dtype=torch.bool, device=dev)
    with torch.no_grad(): lg = model(torch.tensor([ids], device=dev)).logits[0, -1].float()
    lp = torch.log_softmax(lg, -1); return float(-(lp.exp() * lp).sum() / np.log(2))
q = [r for r in ROLL if r["prompt"] == "what shall i do today"][0]["prompt_ids"]
print(f"rig: clean H1 {h1(q, False):.4f} (0.2366) | direction H1 {h1(q, True):.3f} (phase 3: {DIR['hf_H1_per_prompt']['what shall i do today']:.3f})")
t0 = time.time(); meta_rows = []
for s0 in range(0, len(ROLL), SHARD):
    chunk = ROLL[s0:s0 + SHARD]; X = np.full((len(chunk), len(LAYERS), len(SLOTS), D), np.nan, dtype=np.float16)
    for b0 in range(0, len(chunk), BS):
        rows = chunk[b0:b0 + BS]; seqs = [list(r["prompt_ids"]) + list(r["ids"][:K]) for r in rows]; T = max(len(s) for s in seqs)
        ids = torch.full((len(rows), T), EOS, dtype=torch.long, device=dev); att = torch.zeros((len(rows), T), dtype=torch.long, device=dev); m = torch.zeros((len(rows), T), dtype=torch.bool, device=dev)
        for j, (r, s) in enumerate(zip(rows, seqs)):
            ids[j, :len(s)] = torch.tensor(s, device=dev); att[j, :len(s)] = 1; m[j, :len(r["prompt_ids"])] = True
        STATE["on"] = True; STATE["mask"] = m           # direction on prompt positions for BOTH arms' extraction? no: clean arm reads clean
        with torch.no_grad():
            # clean rows get no push (their rollouts were generated without it); direction rows get the push on prompt positions
            for j, r in enumerate(rows):
                if r["arm"] == "clean": m[j] = False
            out = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
        for j, r in enumerate(rows):
            P0 = len(r["prompt_ids"]) - 1; n = min(len(r["ids"]), K)
            for li, L in enumerate(LAYERS):
                hs = out.hidden_states[L][j]
                X[b0 + j, li, 0] = hs[P0].float().cpu().numpy().astype(np.float16)
                for k in range(1, n + 1): X[b0 + j, li, k] = hs[P0 + k].float().cpu().numpy().astype(np.float16)
        if (b0 // BS) % 20 == 0: print(f"  shard {s0//SHARD} batch {b0//BS} {time.time()-t0:.0f}s", flush=True)
    np.save(f"/content/brrrt/features/X_{s0//SHARD:03d}.npy", X); print(f"shard {s0//SHARD}: {X.shape} nan-slots {int(np.isnan(X[:, 0, :, 0]).sum())} {time.time()-t0:.0f}s", flush=True)
json.dump(dict(model=MODEL_ID, layers=LAYERS, slots=SLOTS, K=K, hidden=D, shard=SHARD, eps=EPS, note="hidden_states[L] = residual after block L (0 = embeddings; 36 = after final norm). Direction applied on prompt positions for probe_l0 rows only. Slot R_k = position that has seen k response tokens.",
               rollouts=[dict(arm=r["arm"], prompt_idx=r["prompt_idx"], seed=r["seed"], i=r["i"], n_out=r["n_out"]) for r in ROLL]), open("/content/brrrt/features/features_meta.json", "w"))
print("done", time.time() - t0)


In [ ]:
# === CELL 5 — upload to Hugging Face (dataset repo, private) =========================================
import os, json, time
from google.colab import userdata
from huggingface_hub import HfApi
tok = userdata.get("HF_TOKEN"); api = HfApi(token=tok); me = api.whoami()["name"]
REPO = f"{me}/phase3-layer0-persona-direction-qwen3-8b"
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
t0 = time.time()
api.upload_folder(folder_path="/content/brrrt", repo_id=REPO, repo_type="dataset", path_in_repo=".", ignore_patterns=["*.log"])
print("uploaded", REPO, f"{time.time()-t0:.0f}s")
print(api.list_repo_files(REPO, repo_type="dataset"))
